# Universal optimization with QQA

One notebook for QQA×SCIP, a one-run Pareto front, mixed-variable black-box optimization, and safe TeX-to-model solving. Runtime artifacts stay in the notebook session; no API credential is written to disk.

In [ ]:
%pip install -q "qqa[scip] @ git+https://github.com/Yuma-Ichikawa/QQA4CO.git"
import networkx as nx
import torch

import qqa

device = "cuda" if torch.cuda.is_available() else "cpu"
qqa.fix_seed(7)
print(device, qqa.__version__)

## 1. QQA exploration + SCIP certification

QQA supplies multiple diverse primal starts. SCIP improves the incumbent and returns a dual bound and optimality gap.

In [ ]:
graph = nx.random_regular_graph(3, 30, seed=7)
maxcut = qqa.MaxCut(graph, device=device)
hybrid = qqa.solve_qqa_scip(
    maxcut,
    qqa_kwargs={"sol_size": 256, "num_epochs": 800, "device": device, "verbose": False},
    max_warm_starts=32,
    time_limit=30,
)
print(
    {
        "objective": hybrid.best_obj,
        "status": hybrid.scip_status,
        "gap": hybrid.gap,
        "dual_bound": hybrid.dual_bound,
        "warm_starts": hybrid.n_warm_starts,
    }
)

## 2. One GPU run for a mixed-variable Pareto front

The example trades cost against service quality in a small capacity plan. Augmented Tchebycheff scalarization assigns a different reference direction to every replica.

In [ ]:
planning = qqa.MultiObjectiveProblem(
    [
        qqa.Binary("open", size=2),
        qqa.Integer("units", 0, 10, size=2),
        qqa.Real("reserve", 0.0, 5.0),
    ],
    [
        qqa.Objective(
            lambda v: 12 * v["open"].sum(-1) + 2 * v["units"].sum(-1) + v["reserve"].square(),
            "cost",
            unit="kUSD",
        ),
        qqa.Objective(
            lambda v: 4 * v["units"].sum(-1) + v["reserve"],
            "service",
            direction="max",
        ),
    ],
    constraints=[
        qqa.Constraint(
            lambda v: (v["units"] - 10 * v["open"]).clamp_min(0).sum(-1),
            sense="<=",
            rhs=0,
            weight=500,
            name="activation",
        )
    ],
)
pareto = planning.solve_pareto(sol_size=512, num_epochs=800, device=device, seed=7)
display(pareto.to_frame().head())
qqa.plot_pareto(pareto)

## 3. Budgeted black-box optimization

The objective could call a simulator or remote service. It receives a plain Python dictionary and needs no gradient.

In [ ]:
def simulator(point):
    return (
        4 * (point["enabled"] - 1) ** 2
        + (point["workers"] - 6) ** 2
        + 20 * (point["threshold"] - 0.35) ** 2
    )


tuning = qqa.BlackBoxProblem(
    [
        qqa.Binary("enabled"),
        qqa.Integer("workers", 1, 16),
        qqa.Real("threshold", 0.0, 1.0),
    ],
    simulator,
    constraints=[
        qqa.BlackBoxConstraint(
            lambda p: p["workers"] * p["threshold"], sense="<=", rhs=4.0, name="load"
        )
    ],
)
blackbox = tuning.solve(budget=60, batch_size=6, workers=6, device=device, seed=7)
print(blackbox.best_point, blackbox.best_value, blackbox.feasible)
qqa.plot_blackbox(blackbox)

## 4. TeX → audited JSON → QQA

Enter the key only into the hidden prompt. The client reads it from the process environment. Use `--dry-run` or inspect `translated.spec.to_json()` before solving in a production workflow.

In [ ]:
import getpass
import os

if "QQA_LLM_API_KEY" not in os.environ:
    os.environ["QQA_LLM_API_KEY"] = getpass.getpass("OpenAI-compatible API key: ")

client = qqa.OpenAICompatibleClient(
    # Override base_url/model here for another OpenAI-compatible endpoint.
    # The configured private development gateway uses a non-standard CA.
    # Keep True for ordinary public endpoints.
    verify_ssl=False,
)
tex = r"\min_{x\in[-5,5],\;n\in\{0,\ldots,6\}} (x-2)^2 + (n-3)^2"
spec = qqa.compile_tex(tex, client=client)
print(spec.to_json())
problem = qqa.problem_from_spec(spec)
answer = problem.solve(sol_size=128, num_epochs=600, device=device, verbose=False)
print(answer.score)

For reproducibility without another API call, save only the credential-free spec and later run `qqa tex --spec audited-model.json`. Never store the API key in the notebook.